This is a companion notebook for the book [Deep Learning with Python, Third Edition](https://www.manning.com/books/deep-learning-with-python-third-edition). For readability, it only contains runnable code blocks and section titles, and omits everything else in the book: text paragraphs, figures, and pseudocode.

**If you want to be able to follow what's going on, I recommend reading the notebook side by side with your copy of the book.**

The book's contents are available online at [deeplearningwithpython.io](https://deeplearningwithpython.io).

In [ ]:
!pip install torch torchvision --upgrade -q

## Interpreting what ConvNets learn

### Visualizing intermediate activations

In [ ]:
# PyTorch: chapter 8 was ported to PyTorch, so the saved model is a state_dict
# (e.g. "convnet_from_scratch_with_augmentation.pt") rather than a .keras file.
# Upload it here; we rebuild the matching nn.Module below and load the weights.
from google.colab import files

files.upload()


In [ ]:
import torch
from torch import nn

# PyTorch: rebuild the chapter-8 "convnet from scratch with augmentation"
# architecture as an nn.Module (Conv2D uses valid padding -> padding=0, NCHW).
# The final sigmoid is dropped; the network outputs a single logit.
class ConvNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3), nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 64, kernel_size=3), nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(64, 128, kernel_size=3), nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(128, 256, kernel_size=3), nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(256, 512, kernel_size=3), nn.ReLU(),
        )
        self.dropout = nn.Dropout(0.25)
        self.classifier = nn.Linear(512, 1)

    def forward(self, x):
        # PyTorch: Keras Rescaling(1/255) lived inside the model; do it here.
        x = x / 255.0
        x = self.features(x)
        x = x.mean(dim=(2, 3))  # GlobalAveragePooling2D
        x = self.dropout(x)
        return self.classifier(x)

model = ConvNet()
model.load_state_dict(torch.load("convnet_from_scratch_with_augmentation.pt"))
model.eval()
print(model)


In [ ]:
# PyTorch: no keras.utils helpers; download with torchvision and load with PIL.
# We return a NCHW float32 tensor (the layout torch convs expect).
import numpy as np
from PIL import Image
from torchvision.datasets.utils import download_url

download_url("https://img-datasets.s3.amazonaws.com/cat.jpg", ".", "cat.jpg")
img_path = "cat.jpg"

def get_img_array(img_path, target_size):
    img = Image.open(img_path).convert("RGB").resize(target_size)
    array = np.asarray(img, dtype="float32")          # HWC
    array = np.transpose(array, (2, 0, 1))            # PyTorch: -> CHW
    array = np.expand_dims(array, axis=0)             # -> NCHW
    return torch.from_numpy(array)

img_tensor = get_img_array(img_path, target_size=(180, 180))


In [ ]:
import matplotlib.pyplot as plt

plt.axis("off")
# PyTorch: img_tensor is NCHW; move channels last and to uint8 for display.
plt.imshow(img_tensor[0].permute(1, 2, 0).numpy().astype("uint8"))
plt.show()


In [ ]:
# PyTorch: instead of building a new model that returns layer outputs, we
# register forward hooks on every Conv2d / MaxPool2d to capture their activations.
layer_names = []
_activation_store = []
_hooks = []

def _make_hook():
    def hook(module, inp, out):
        _activation_store.append(out.detach())
    return hook

for name, module in model.features.named_children():
    if isinstance(module, (nn.Conv2d, nn.MaxPool2d)):
        layer_names.append(f"features.{name} ({type(module).__name__})")
        _hooks.append(module.register_forward_hook(_make_hook()))


In [ ]:
# PyTorch: run a forward pass; the hooks fill _activation_store in order.
_activation_store.clear()
with torch.no_grad():
    model(img_tensor)
activations = [a.numpy() for a in _activation_store]


In [ ]:
first_layer_activation = activations[0]
print(first_layer_activation.shape)  # PyTorch: NCHW, channels are axis 1


In [ ]:
import matplotlib.pyplot as plt

# PyTorch: channels-first, so channel 5 is [0, 5, :, :] instead of [0, :, :, 5].
plt.matshow(first_layer_activation[0, 5, :, :], cmap="viridis")


In [ ]:
images_per_row = 16
for layer_name, layer_activation in zip(layer_names, activations):
    # PyTorch: activations are NCHW, so features = axis 1 and the spatial size = axis 2.
    n_features = layer_activation.shape[1]
    size = layer_activation.shape[2]
    n_cols = n_features // images_per_row
    display_grid = np.zeros(
        ((size + 1) * n_cols - 1, images_per_row * (size + 1) - 1)
    )
    for col in range(n_cols):
        for row in range(images_per_row):
            channel_index = col * images_per_row + row
            channel_image = layer_activation[0, channel_index, :, :].copy()
            if channel_image.sum() != 0:
                channel_image -= channel_image.mean()
                channel_image /= channel_image.std()
                channel_image *= 64
                channel_image += 128
            channel_image = np.clip(channel_image, 0, 255).astype("uint8")
            display_grid[
                col * (size + 1) : (col + 1) * size + col,
                row * (size + 1) : (row + 1) * size + row,
            ] = channel_image
    scale = 1.0 / size
    plt.figure(
        figsize=(scale * display_grid.shape[1], scale * display_grid.shape[0])
    )
    plt.title(layer_name)
    plt.grid(False)
    plt.axis("off")
    plt.imshow(display_grid, aspect="auto", cmap="viridis")


### Visualizing ConvNet filters

In [ ]:
import torchvision
from torchvision.models import resnet50, ResNet50_Weights

# PyTorch: keras_hub has no Xception preset here. We substitute torchvision's
# ImageNet-pretrained ResNet50 as the convnet backbone. Its standard preprocessing
# transform replaces the keras_hub ImageConverter (resize to 180x180 + normalize).
weights = ResNet50_Weights.IMAGENET1K_V2
model = resnet50(weights=weights)
model.eval()

_resnet_transform = weights.transforms()  # the model's expected preprocessing

def preprocessor(image_nchw):
    # image is a NCHW float tensor in [0, 255]; resize to 180x180 and normalize
    # the same way the pretrained weights expect.
    import torch.nn.functional as F
    x = F.interpolate(image_nchw, size=(180, 180), mode="bilinear", align_corners=False)
    mean = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1)
    std = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1)
    return (x / 255.0 - mean) / std


In [ ]:
# PyTorch: list the convolutional layers by walking named_modules().
for name, module in model.named_modules():
    if isinstance(module, nn.Conv2d):
        print(name)


In [ ]:
# PyTorch: extract one layer's output via a forward hook. We pick an intermediate
# conv block of ResNet50 (analogous to an early/mid separable-conv block).
layer_name = "layer2.0.conv2"
_target_layer = dict(model.named_modules())[layer_name]
_fx_store = {}

def _fx_hook(module, inp, out):
    _fx_store["activation"] = out

_target_layer.register_forward_hook(_fx_hook)

def feature_extractor(image):
    model(image)
    return _fx_store["activation"]


In [0]:
activation = feature_extractor(preprocessor(img_tensor))

In [ ]:
# PyTorch: keras.ops -> torch; activation is NCHW so the filter is axis 1 and
# we crop the spatial axes (2, 3).
def compute_loss(image, filter_index):
    activation = feature_extractor(image)
    filter_activation = activation[:, filter_index, 2:-2, 2:-2]
    return torch.mean(filter_activation)


#### Gradient ascent in PyTorch

In [ ]:
# PyTorch: enable grad on the input, backprop the loss, then step along the
# (normalized) gradient. ops.normalize -> divide by the L2 norm.
def gradient_ascent_step(image, filter_index, learning_rate):
    image = image.clone().detach().requires_grad_(True)
    loss = compute_loss(image, filter_index)
    loss.backward()
    grads = image.grad
    grads = grads / (torch.sqrt(torch.mean(grads ** 2)) + 1e-8)
    image = image + learning_rate * grads
    return image.detach()


#### The filter visualization loop

In [ ]:
def generate_filter_pattern(filter_index):
    iterations = 30
    learning_rate = 10.0
    # PyTorch: keras.random.uniform -> torch.rand; shape is NCHW.
    image = 0.4 + 0.2 * torch.rand(1, 3, img_width, img_height)
    for i in range(iterations):
        image = gradient_ascent_step(image, filter_index, learning_rate)
    return image[0]


In [ ]:
def deprocess_image(image):
    # PyTorch: keras.ops -> torch; input is CHW, output is a HWC uint8 NumPy array.
    image = image - torch.mean(image)
    image = image / (torch.std(image) + 1e-8)
    image = image * 64
    image = image + 128
    image = torch.clip(image, 0, 255)
    image = image.permute(1, 2, 0)        # CHW -> HWC
    image = image[25:-25, 25:-25, :]
    image = image.to(torch.uint8)
    return image.numpy()


In [0]:
plt.axis("off")
plt.imshow(deprocess_image(generate_filter_pattern(filter_index=2)))

In [ ]:
all_images = []
for filter_index in range(64):
    print(f"Processing filter {filter_index}")
    image = deprocess_image(generate_filter_pattern(filter_index))
    all_images.append(image)

margin = 5
n = 8
box_width = img_width - 25 * 2
box_height = img_height - 25 * 2
full_width = n * box_width + (n - 1) * margin
full_height = n * box_height + (n - 1) * margin
stitched_filters = np.zeros((full_width, full_height, 3))

for i in range(n):
    for j in range(n):
        image = all_images[i * n + j]
        stitched_filters[
            (box_width + margin) * i : (box_width + margin) * i + box_width,
            (box_height + margin) * j : (box_height + margin) * j + box_height,
            :,
        ] = image

# PyTorch: keras.utils.save_img -> matplotlib's imsave (expects uint8 HWC).
plt.imsave(f"filters_for_layer_{layer_name}.png", stitched_filters.astype("uint8"))


### Visualizing heatmaps of class activation

In [ ]:
# PyTorch: download + load with torchvision/PIL instead of keras.utils.
from torchvision.datasets.utils import download_url

download_url(
    "https://img-datasets.s3.amazonaws.com/elephant.jpg", ".", "elephant.jpg"
)
img_path = "elephant.jpg"
img = Image.open(img_path).convert("RGB")
img_array = np.expand_dims(np.asarray(img, dtype="float32"), axis=0)  # NHWC, for display


In [ ]:
# PyTorch: the pretrained ImageClassifier is the same torchvision ResNet50
# we loaded above (1000 ImageNet classes). We preprocess to NCHW, run a forward
# pass, and apply softmax ourselves to get probabilities.
import torch.nn.functional as F

img_nchw = torch.from_numpy(np.transpose(img_array, (0, 3, 1, 2)))  # NHWC -> NCHW
with torch.no_grad():
    preds = F.softmax(model(preprocessor(img_nchw)), dim=1).numpy()
preds.shape


In [ ]:
# PyTorch: decode with the ImageNet class names that ship with the weights.
categories = weights.meta["categories"]
top5 = preds[0].argsort()[-5:][::-1]
[(categories[i], float(preds[0][i])) for i in top5]


In [0]:
np.argmax(preds[0])

In [ ]:
# PyTorch: apply the same preprocessing transform (works on the NCHW tensor).
img_array = preprocessor(img_nchw)


In [ ]:
# PyTorch: Grad-CAM needs the last conv feature map. We hook the final conv
# block of ResNet50 (layer4) and split the network into "up to last conv" and
# "classifier head" so we can take gradients w.r.t. the feature map.
last_conv_layer_name = "layer4"
_last_conv_store = {}

def _last_conv_hook(module, inp, out):
    _last_conv_store["out"] = out

dict(model.named_modules())[last_conv_layer_name].register_forward_hook(_last_conv_hook)

def last_conv_layer_model(image):
    model(image)
    return _last_conv_store["out"]


In [ ]:
# PyTorch: the classifier head on top of layer4's feature map is ResNet50's
# avgpool + fc. This maps a (N, 2048, H, W) feature map to class logits.
def classifier_model(feature_map):
    x = model.avgpool(feature_map)
    x = torch.flatten(x, 1)
    return model.fc(x)


#### Getting the gradient of the top class: PyTorch version

In [ ]:
# PyTorch: capture the last conv feature map, make it require grad, run the
# classifier head, and backprop the top-class logit to get the gradient.
def get_top_class_gradients(img_array):
    last_conv_layer_output = last_conv_layer_model(img_array)
    last_conv_layer_output = (
        last_conv_layer_output.clone().detach().requires_grad_(True)
    )
    preds = classifier_model(last_conv_layer_output)
    top_pred_index = torch.argmax(preds[0])
    top_class_channel = preds[:, top_pred_index]
    top_class_channel.backward()
    grads = last_conv_layer_output.grad
    return grads, last_conv_layer_output

grads, last_conv_layer_output = get_top_class_gradients(img_array)
# PyTorch: feature maps are NCHW; move to NHWC NumPy so the heatmap code below
# (which averages over spatial axes 0,1,2 and indexes channels last) works.
grads = grads.permute(0, 2, 3, 1).detach().numpy()
last_conv_layer_output = last_conv_layer_output.permute(0, 2, 3, 1).detach().numpy()


#### Displaying the class activation heatmap

In [0]:
pooled_grads = np.mean(grads, axis=(0, 1, 2))
last_conv_layer_output = last_conv_layer_output[0].copy()
for i in range(pooled_grads.shape[-1]):
    last_conv_layer_output[:, :, i] *= pooled_grads[i]
heatmap = np.mean(last_conv_layer_output, axis=-1)

In [0]:
heatmap = np.maximum(heatmap, 0)
heatmap /= np.max(heatmap)
plt.matshow(heatmap)

In [ ]:
import matplotlib.cm as cm

# PyTorch: keras.utils image helpers -> PIL.
img = np.asarray(Image.open(img_path).convert("RGB"), dtype="float32")

heatmap = np.uint8(255 * heatmap)

jet = cm.get_cmap("jet")
jet_colors = jet(np.arange(256))[:, :3]
jet_heatmap = jet_colors[heatmap]

jet_heatmap = Image.fromarray((jet_heatmap * 255).astype("uint8"))
jet_heatmap = jet_heatmap.resize((img.shape[1], img.shape[0]))
jet_heatmap = np.asarray(jet_heatmap, dtype="float32")

superimposed_img = jet_heatmap * 0.4 + img
superimposed_img = Image.fromarray(np.clip(superimposed_img, 0, 255).astype("uint8"))

plt.imshow(superimposed_img)


### Visualizing the latent space of a ConvNet